# Use Case 5: Improve CNN Generalization with Data Augmentation

This notebook applies random flip, rotation, zoom and contrast changes during training.

Each code cell is preceded by Markdown that explains what the step does, why it is needed, and which deep-learning concept is being demonstrated.


## Step 1: Import libraries

Random augmentation layers are imported in addition to CNN layers.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)

from tensorflow.keras.layers import (
    RandomFlip,
    RandomRotation,
    RandomZoom,
    RandomContrast,
)


## Step 2: Configure and load data

The same real multi-class dataset is used.

In [ ]:
DATASET_PATH = "../datasets/02_multiclass_objects"
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 16

DATASET_DIR = Path(DATASET_PATH)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))


## Step 3: Create an augmentation pipeline

These transformations produce different image variations each time the training batch passes through the model.

In [ ]:
augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1),
    RandomContrast(0.1),
])

images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for index in range(9):
    augmented = augmentation(images[:1], training=True)

    plt.subplot(3, 3, index + 1)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.axis("off")

plt.tight_layout()
plt.show()


## Step 4: Build the augmented CNN

Augmentation is placed before normalization and convolution.

In [ ]:
model = Sequential([
    Input(shape=(128, 128, 3)),
    augmentation,
    Rescaling(1.0 / 255),

    Conv2D(32, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(64, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    Conv2D(128, 3, activation="relu", padding="same"),
    MaxPooling2D(),

    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.4),
    Dense(len(class_names), activation="softmax"),
])


## Step 5: Compile and train

The model sees varied images in every epoch, which can reduce overfitting.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
)


## Step 6: Evaluate

Compare the gap between training and validation accuracy.

In [ ]:
loss, accuracy = model.evaluate(val_ds)

print("Validation loss:", round(loss, 4))
print("Validation accuracy:", round(accuracy, 4))

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()
